In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns
import pandas as pd

# ==================== Config ====================
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'
models_to_compare = ['stella_en_400M_v5']   # specify model(s)
analyz_split_win_size = 2                   # window size
output_dir = os.path.join(folder_path, "heatmap_results")
os.makedirs(output_dir, exist_ok=True)

# ==================== Font (English only) ====================
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ==================== Load data ====================
json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
all_models_data = {}

for file in json_files:
    model_name = file.replace(".json", "")
    if models_to_compare and model_name not in models_to_compare:
        continue
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    all_models_data[model_name] = data

if not all_models_data:
    raise ValueError("❌ No matching models found!")

# ==================== Tasks ====================
sample_model = next(iter(all_models_data.values()))
tasks = [
        'AmazonCounterfactualClassification',
        'AmazonReviewsClassification', 'Banking77Classification',
        'EmotionClassification', 'ImdbClassification',
        'MTOPDomainClassification', 'MTOPIntentClassification',
        'MassiveIntentClassification', 'MassiveScenarioClassification',
        'ToxicConversationsClassification', 'TweetSentimentExtractionClassification',
        'BiorxivClusteringS2S',
        'MedrxivClusteringS2S',
        'TwentyNewsgroupsClustering',
        'SprintDuplicateQuestions', 'TwitterSemEval2015', 'TwitterURLCorpus',
        'AskUbuntuDupQuestions', 'SciDocsRR', 'StackOverflowDupQuestions',
        'BIOSSES', 'SICK-R', 'STS12', 'STS13', 'STS14', 'STS15',
        'STS16', 'STS17', 'STSBenchmark',
        'ArguAna', 'CQADupstackEnglishRetrieval',
        'NFCorpus', 'SCIDOCS', 'SciFact'
    ]

# ==================== Build normalized matrix ====================
for model_name, data in all_models_data.items():
    print(f"\n=== Processing model: {model_name} ===")

    normalized_dict = {}
    max_len = 0

    for task_name in tasks:
        task_data = data["task_name"][task_name]
        split_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
        if not split_data:
            print(f"⚠️ {model_name} missing split_win_size={analyz_split_win_size} data for {task_name}")
            continue

        chunk_result_data = np.array(split_data['chunk_result'], dtype=float)
        n = len(chunk_result_data)
        if n == 0:
            continue

        # === sort & normalize ===
        sorted_indices = np.argsort(chunk_result_data)
        normalized_indices = np.arange(n) / n
        sorted_values = normalized_indices[sorted_indices]

        normalized_dict[task_name] = sorted_values
        max_len = max(max_len, n)

    if not normalized_dict:
        print(f"⚠️ No valid tasks for {model_name}")
        continue

    # === pad all sequences to same length ===
    matrix = []
    for task_name in tasks:
        seq = normalized_dict.get(task_name, [])
        if len(seq) == 0:
            seq = np.full(max_len, np.nan)
        elif len(seq) < max_len:
            seq = np.pad(seq, (0, max_len - len(seq)), constant_values=np.nan)
        matrix.append(seq)

    matrix = np.array(matrix)
    df = pd.DataFrame(matrix, index=tasks)

    # ==================== Plot combined heatmap ====================
    fig_width = max_len / 120 if max_len > 1200 else 12
    fig_height = max(len(tasks) * 0.05, 8)

    plt.figure(figsize=(fig_width, fig_height))
    sns.heatmap(
        df,
        cmap="viridis",
        cbar=True,
        xticklabels=False,
        yticklabels=True,
        square=False,
    )

    plt.title(f"Model: {model_name} (split_win_size={analyz_split_win_size})", fontsize=12, pad=10)
    plt.ylabel("Task", fontsize=10)
    plt.xlabel("Chunk index (sorted & normalized)", fontsize=9)
    plt.tick_params(axis='y', labelsize=8, pad=8)
    plt.tight_layout(rect=[0.0, 0, 1, 1])

    output_file = os.path.join(output_dir, f"{model_name}_all_tasks_heatmap_matrix.png")
    plt.savefig(output_file, dpi=200)
    plt.close()

    print(f"✅ Saved: {output_file}")

print(f"\n🎨 All combined matrix heatmaps have been saved to: {output_dir}")


In [ ]:
# 绘制不同模型的拟合结果
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from matplotlib.colors import LinearSegmentedColormap
from scipy import interpolate
from scipy.interpolate import interp1d

# 指定文件夹路径
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'

# 获取所有文件和文件夹名称
files = os.listdir(folder_path)

# 筛选出以.json结尾的文件
json_files = [f for f in files if f.endswith('.json')]

# 选择要比较的模型（可以根据需要修改这个列表）
models_to_compare = ['stella_en_400M_v5-GEDI']  # 示例模型名称

# 存储所有模型的数据
all_models_data = {}

# 选择需要分析的最小切割窗口
analyz_split_win_size = 1

# 读取所有模型数据
for file in json_files:
    model_name = file.replace(".json", "")
    
    # 如果指定了要比较的模型列表，则只处理这些模型
    if models_to_compare and model_name in models_to_compare:
        continue
        
    print(f"Processing {model_name}")
    
    # 读取文件
    file_path = folder_path + "{}.json".format(model_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    all_models_data[model_name] = data

# 提取任务信息（假设所有模型有相同的任务）
tasks = list(all_models_data[list(all_models_data.keys())[0]]["task_name"].keys())



for model_name, data in all_models_data.items():
    # 为模型创建综合图表
    for task_name in tasks:
        
        task_data = data["task_name"][task_name]
        
        # 获取split_win_size的数据
        split_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
        if not split_data:
            print(f"Warning: {model_name} has no split_win_size={analyz_split_win_size} data for {task_name}")
            continue

        chunk_result_data = split_data['chunk_result']

        # 计算chunk_result_data总长度，然后根据chunk_result_data的数值大小，计算出一个从小到大的排序index。比如chunk_result_data中的数据是[0.1, 0.3, 0.2]，得到的Index排序就是[0,2,1]
        # 然后对这个Index序列，除以chunk_result_data总长度，归一化后，画一个热力图单独保存。同时也把所有的热力图拼在一个大图里，共同展示，弄成一个大热力图。
            
        


In [ ]:
# 绘制不同模型的拟合结果（按任务类别分类）
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from matplotlib.colors import LinearSegmentedColormap
from scipy import interpolate
from scipy.interpolate import interp1d

# 任务分类表
task_categories = {
    "Classification": [
        'AmazonCounterfactualClassification',
        'AmazonReviewsClassification', 'Banking77Classification',
        'EmotionClassification', 'ImdbClassification',
        'MTOPDomainClassification', 'MTOPIntentClassification',
        'MassiveIntentClassification', 'MassiveScenarioClassification',
        'ToxicConversationsClassification', 'TweetSentimentExtractionClassification'
    ],
    "Clustering": [
        'BiorxivClusteringS2S',
        'MedrxivClusteringS2S',
        'TwentyNewsgroupsClustering'
    ],
    "PairClassification": [
        'SprintDuplicateQuestions', 'TwitterSemEval2015', 'TwitterURLCorpus'
    ],
    "Reranking": [
        'AskUbuntuDupQuestions', 'SciDocsRR', 'StackOverflowDupQuestions'
    ],
    "STS": [
        'BIOSSES', 'SICK-R', 'STS12', 'STS13', 'STS14', 'STS15',
        'STS16', 'STS17', 'STSBenchmark'
    ],
    "Retrieval": [
        'ArguAna', 'CQADupstackEnglishRetrieval',
        'NFCorpus', 'SCIDOCS', 'SciFact'
    ],
    "MTEB": [
        'AmazonCounterfactualClassification',
        'AmazonReviewsClassification', 'Banking77Classification',
        'EmotionClassification', 'ImdbClassification',
        'MTOPDomainClassification', 'MTOPIntentClassification',
        'MassiveIntentClassification', 'MassiveScenarioClassification',
        'ToxicConversationsClassification', 'TweetSentimentExtractionClassification',
        'BiorxivClusteringS2S',
        'MedrxivClusteringS2S',
        'TwentyNewsgroupsClustering',
        'SprintDuplicateQuestions', 'TwitterSemEval2015', 'TwitterURLCorpus',
        'AskUbuntuDupQuestions', 'SciDocsRR', 'StackOverflowDupQuestions',
        'BIOSSES', 'SICK-R', 'STS12', 'STS13', 'STS14', 'STS15',
        'STS16', 'STS17', 'STSBenchmark',
        'ArguAna', 'CQADupstackEnglishRetrieval',
        'NFCorpus', 'SCIDOCS', 'SciFact'
    ],
}


    # "Summarization": [
    #     'SummEval'
    # ],

# 指定文件夹路径
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'

# 获取所有文件和文件夹名称
files = os.listdir(folder_path)

# 筛选出以.json结尾的文件
json_files = [f for f in files if f.endswith('.json')]

# 选择要比较的模型（可以根据需要修改这个列表）
models_to_compare = ['gtr-t5-large.json', 'bart-base.json', 'stella_en_400M_v5.json', 'mxbai-embed-large-v1.json']  # 示例模型名称

# 存储所有模型的数据
all_models_data = {}

# 选择需要分析的最小切割窗口
analyz_split_win_size = 2

# 读取所有模型数据
for file in json_files:
# for file in models_to_compare:
    model_name = file.replace(".json", "")
    
    print(f"Processing {model_name}")
    
    # 读取文件
    file_path = folder_path + "{}.json".format(model_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    all_models_data[model_name] = data

# 定义Score Types的颜色和标签
score_colors = {
    'head_score': '#1f77b4',    # 蓝色 - Best Chunks
    'end_score': '#ff7f0e',     # 橙色 - Poor Chunks  
    'random_score': '#2ca02c',  # 绿色 - Random Chunks
    'sort_score': '#d62728'     # 红色 - Sort Chunks
}

score_labels = {
    'head_score': 'Best Chunks',
    'end_score': 'Poor Chunks',
    'random_score': 'Random Chunks',
    'sort_score': 'Sort Chunks'
}

# 定义模型的marks和line样式组合
model_styles = {
    'markers': ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'X', 'd'],
    'linestyles': ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]
}

# 为每个任务类别创建综合图表
for category_name, category_tasks in task_categories.items():
    print(f"\nProcessing category: {category_name}")
    print(f"Tasks in category: {category_tasks}")
    
    plt.figure(figsize=(12, 8))
    
    # 存储所有模型的归一化数据用于拟合（按类别聚合）
    all_normalized_data = {
        'head_score': [],
        'end_score': [],
        'random_score': [],
        'sort_score': []
    }
    
    # 存储每个模型的random_min值用于归一化（按类别聚合）
    random_mins = {}
    
    # 首先检查该类别下哪些任务实际存在于数据中
    available_tasks = []
    for model_name, data in all_models_data.items():
        model_tasks = list(data["task_name"].keys())
        available_tasks.extend([task for task in category_tasks if task in model_tasks])
    
    available_tasks = list(set(available_tasks))  # 去重
    print(f"Available tasks in data: {available_tasks}")
    
    if not available_tasks:
        print(f"No available tasks found for category {category_name}, skipping...")
        plt.close()
        continue
    
    # 首先收集所有模型的random_min值（按类别平均）
    for model_name, data in all_models_data.items():
        category_random_mins = []
        
        for task_name in available_tasks:
            if task_name not in data["task_name"]:
                continue
                
            task_data = data["task_name"][task_name]
            baseline = task_data["defult_score"]
            
            # 获取split_win_size=8的数据
            split_8_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
            if not split_8_data:
                continue
                
            # 获取random_score的最小值
            if "chunk_win_size" in split_8_data:
                random_scores = []
                for chunk_size, score_dict in split_8_data["chunk_win_size"].items():
                    random_scores.append(score_dict["random_score"]["main_score"])
                
                if random_scores:
                    category_random_mins.append(min(random_scores))
        
        if category_random_mins:
            random_mins[model_name] = np.mean(category_random_mins)
        else:
            random_mins[model_name] = 0
    
    # 为每个模型分配独特的marks和line组合
    model_style_map = {}
    for i, model_name in enumerate(all_models_data.keys()):
        marker_idx = i % len(model_styles['markers'])
        linestyle_idx = (i // len(model_styles['markers'])) % len(model_styles['linestyles'])
        
        model_style_map[model_name] = {
            'marker': model_styles['markers'][marker_idx],
            'linestyle': model_styles['linestyles'][linestyle_idx],
            'markersize': 3,
            'linewidth': 1,
            'alpha': 0.1  # 原始数据稍微淡化
        }
    
    # 绘制每个模型的原始数据（按类别聚合）
    for model_name, data in all_models_data.items():
        # 收集该类别下所有任务的数据
        category_records = []
        
        for task_name in available_tasks:
            if task_name not in data["task_name"]:
                continue
                
            task_data = data["task_name"][task_name]
            baseline = task_data["defult_score"]
            
            # 获取split_win_size=8的数据
            split_8_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
            if not split_8_data:
                continue
                
            # 收集数据
            if "chunk_win_size" in split_8_data:
                for chunk_size, score_dict in split_8_data["chunk_win_size"].items():
                    category_records.append({
                        "chunk_win_size": int(chunk_size),
                        "head_score": score_dict["head_score"]["main_score"],
                        "end_score": score_dict["end_score"]["main_score"],
                        "random_score": score_dict["random_score"]["main_score"],
                        "sort_score": score_dict["sort_score"]["main_score"],
                        "baseline": baseline,
                        "task": task_name
                    })
        
        if not category_records:
            continue
            
        # 转换为DataFrame
        df = pd.DataFrame(category_records)
        
        # 按chunk_win_size分组计算平均值（跨任务平均）
        df_avg = df.groupby("chunk_win_size").agg({
            "head_score": "mean",
            "end_score": "mean", 
            "random_score": "mean",
            "sort_score": "mean",
            "baseline": "mean"
        }).reset_index().sort_values("chunk_win_size")
        
        # 计算平均baseline
        avg_baseline = df["baseline"].mean()
        
        # 归一化处理
        random_min = random_mins[model_name]
        normalization_range = avg_baseline - random_min if avg_baseline > random_min else 1
        
        # 归一化公式：归一化值 = (原始值 - random_min) / normalization_range
        df_avg["head_score_norm"] = (df_avg["head_score"] - random_min) / normalization_range
        df_avg["end_score_norm"] = (df_avg["end_score"] - random_min) / normalization_range
        df_avg["random_score_norm"] = (df_avg["random_score"] - random_min) / normalization_range
        df_avg["sort_score_norm"] = (df_avg["sort_score"] - random_min) / normalization_range
        
        # 存储归一化数据用于拟合
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            all_normalized_data[score_type].append({
                'chunk_sizes': df_avg["chunk_win_size"].values,
                'scores': df_avg[f"{score_type}_norm"].values,
                'model': model_name
            })
        
        # 获取模型的样式
        style = model_style_map[model_name]
        
        # 绘制原始数据线条（保留详细样式）
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            plt.plot(df_avg["chunk_win_size"], df_avg[f"{score_type}_norm"], 
                    color=score_colors[score_type],
                    marker=style['marker'],
                    linestyle=style['linestyle'],
                    markersize=style['markersize'],
                    linewidth=style['linewidth'],
                    alpha=style['alpha'],
                    label=f"{model_name} - {score_labels[score_type]}")
    
    # 拟合并绘制平均曲线（粗线条显示）
    # 获取所有模型共有的chunk_sizes（取并集）
    all_chunk_sizes = set()
    for score_data in all_normalized_data['head_score']:
        all_chunk_sizes.update(score_data['chunk_sizes'])
    
    if all_chunk_sizes:
        common_chunk_sizes = np.array(sorted(all_chunk_sizes))
        
        # 对每种score类型进行拟合
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            if not all_normalized_data[score_type]:
                continue
                
            # 收集所有数据点用于拟合
            all_x = []
            all_y = []
            
            for model_data in all_normalized_data[score_type]:
                all_x.extend(model_data['chunk_sizes'])
                all_y.extend(model_data['scores'])
            
            # 转换为numpy数组
            all_x = np.array(all_x)
            all_y = np.array(all_y)
            
            # 按chunk_size分组计算平均值和标准差
            unique_sizes = np.unique(all_x)
            means = []
            stds = []
            
            for size in unique_sizes:
                mask = all_x == size
                if np.sum(mask) > 0:
                    means.append(np.mean(all_y[mask]))
                    stds.append(np.std(all_y[mask]))
                else:
                    means.append(np.nan)
                    stds.append(np.nan)
            
            means = np.array(means)
            stds = np.array(stds)
            
            # 移除NaN值
            valid_mask = ~np.isnan(means)
            if np.sum(valid_mask) < 2:
                continue
                
            valid_sizes = unique_sizes[valid_mask]
            valid_means = means[valid_mask]
            valid_stds = stds[valid_mask]
            
            # 使用样条插值进行平滑拟合
            if len(valid_sizes) >= 3:
                # 对x取对数进行插值（因为x轴是对数尺度）
                log_sizes = np.log2(valid_sizes)
                log_common = np.log2(common_chunk_sizes)
                
                # 样条插值
                try:
                    spline = interp1d(log_sizes, valid_means, kind='cubic', 
                                    bounds_error=False, fill_value="extrapolate")
                    fitted_means = spline(log_common)
                    
                    # 计算拟合的标准差（使用最近点的标准差）
                    std_interp = interp1d(log_sizes, valid_stds, kind='linear',
                                        bounds_error=False, fill_value="extrapolate")
                    fitted_stds = std_interp(log_common)
                    
                    # 绘制拟合曲线（粗线条）
                    plt.plot(common_chunk_sizes, fitted_means, 
                            color=score_colors[score_type],
                            linewidth=3,
                            alpha=0.9,
                            label=f"Fitted {score_labels[score_type]}")
                    
                    # 添加置信区间（标准差范围）
                    plt.fill_between(common_chunk_sizes, 
                                   fitted_means - fitted_stds,
                                   fitted_means + fitted_stds,
                                   color=score_colors[score_type], 
                                   alpha=0.1)
                    
                except:
                    # 如果样条插值失败，使用线性插值
                    try:
                        linear_interp = interp1d(log_sizes, valid_means, kind='linear',
                                               bounds_error=False, fill_value="extrapolate")
                        fitted_means = linear_interp(log_common)
                        
                        std_interp = interp1d(log_sizes, valid_stds, kind='linear',
                                            bounds_error=False, fill_value="extrapolate")
                        fitted_stds = std_interp(log_common)
                        
                        plt.plot(common_chunk_sizes, fitted_means, 
                                color=score_colors[score_type],
                                linewidth=3,
                                alpha=0.9,
                                label=f"Fitted {score_labels[score_type]}")
                        
                        plt.fill_between(common_chunk_sizes, 
                                       fitted_means - fitted_stds,
                                       fitted_means + fitted_stds,
                                       color=score_colors[score_type], 
                                       alpha=0.1)
                    except:
                        # 如果插值完全失败，直接绘制原始点
                        plt.plot(valid_sizes, valid_means, 
                                color=score_colors[score_type],
                                linewidth=3,
                                alpha=0.9,
                                label=f"Fitted {score_labels[score_type]}")
                        
                        plt.fill_between(valid_sizes, 
                                       valid_means - valid_stds,
                                       valid_means + valid_stds,
                                       color=score_colors[score_type], 
                                       alpha=0.2)
    
    # 添加统一的基线（100%位置）
    plt.axhline(y=1, color="red", linestyle="--", linewidth=2, alpha=0.8, label="Baseline")
    
    # # 设置x轴为log2
    # plt.xscale("log", base=2)
    
    # 设置x轴刻度
    if all_chunk_sizes:
        xticks = sorted(all_chunk_sizes)
        # plt.xticks(xticks, [f"2^{int(np.log2(x))}" for x in xticks])
        plt.xticks(xticks, xticks)
    
    plt.title(f"{category_name} Category", 
              fontsize=14, fontweight='bold')
    plt.xlabel("Dimensional Size", fontsize=12)
    plt.ylabel("Normalized Score (0% = Random Min, 100% = Baseline)", fontsize=12)
    
    # 创建两个图例：一个用于Score Types（颜色），一个用于模型（marks+line）
    from matplotlib.lines import Line2D
    
    # 第一个图例：Score Types和拟合曲线（颜色）
    score_legend_elements = []
    for score_type, color in score_colors.items():
        score_legend_elements.append(
            Line2D([0], [0], color=color, lw=2, label=f"Fitted {score_labels[score_type]}")
        )
    
    # 第二个图例：模型（marks+line组合）
    model_legend_elements = []
    for model_name, style in model_style_map.items():
        model_legend_elements.append(
            Line2D([0], [0], color='gray', 
                  marker=style['marker'], 
                  linestyle=style['linestyle'],
                  markersize=8,
                  lw=2, 
                  label=model_name)
        )
    
    # 添加基线到图例
    score_legend_elements.append(
        Line2D([0], [0], color='red', linestyle='--', lw=3, alpha=0.7, label='Baseline')
    )
    
    # 创建图例
    legend1 = plt.legend(handles=score_legend_elements, loc='lower left', 
                        bbox_to_anchor=(0.6, 0.02), title="Score Types", fontsize=9, framealpha=0.9)
    legend2 = plt.legend(handles=model_legend_elements, loc='lower left', 
                        bbox_to_anchor=(0.8, 0.02), title="Models", fontsize=9, framealpha=0.9)
    
    # 添加图例到图表
    plt.gca().add_artist(legend1)
    plt.gca().add_artist(legend2)
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # 保存图表
    output_dir = "/home/linkco/exa/Useful-Embedding/data/analyze/category_comparison_visualizations"
    os.makedirs(output_dir, exist_ok=True)
    
    # 替换文件名中的非法字符
    safe_category_name = category_name.replace("/", "_").replace("\\", "_").replace(":", "_")
    save_path = os.path.join(output_dir, f"{safe_category_name}_category_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved category comparison chart for {category_name} to {save_path}")
    
    plt.show()
    plt.close()

print("All category comparison charts generated successfully!")